# Week 15: Unsupervised Learning & Model Evaluation — Applied Statistics with Python (2026)

Last week we learned supervised learning where labels guide the model. This week we explore **unsupervised learning** — algorithms that discover hidden structure in data *without* labels. We also cover advanced model evaluation techniques that apply to both supervised and unsupervised settings.

| # | Topic |
|---|-------|
| 1 | Introduction to Unsupervised Learning |
| 2 | k-Means Clustering |
| 3 | Choosing k: Elbow Method & Silhouette Analysis |
| 4 | Hierarchical Clustering |
| 5 | DBSCAN: Density-Based Clustering |
| 6 | PCA: Principal Component Analysis |
| 7 | PCA for Visualization & Dimensionality Reduction |
| 8 | Advanced Model Evaluation: Bias-Variance Trade-off |
| 9 | Learning Curves & Validation Curves |
| 10 | Practical Example: Customer Segmentation |
| 11 | Summary & Homework |

## Imports

We import scikit-learn's clustering, decomposition, and evaluation modules.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import warnings
warnings.filterwarnings('ignore')

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             adjusted_rand_score, normalized_mutual_info_score,
                             calinski_harabasz_score, davies_bouldin_score)

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Preprocessing & evaluation
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                      learning_curve, validation_curve)
from sklearn.datasets import load_iris, load_wine, make_blobs, make_moons

# Supervised models for evaluation section
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)

print('All imports successful!')

---
## 1. Introduction to Unsupervised Learning

In **unsupervised learning**, we have features X but **no labels y**. The goal is to discover hidden patterns or structure.

| Task | Goal | Algorithms |
|------|------|------------|
| **Clustering** | Group similar data points | k-Means, Hierarchical, DBSCAN |
| **Dimensionality Reduction** | Reduce features while preserving info | PCA, t-SNE |
| **Anomaly Detection** | Find unusual data points | Isolation Forest, LOF |
| **Association Rules** | Find co-occurring items | Apriori, FP-Growth |

### When to Use Unsupervised Learning

- **No labels available** — most real-world data is unlabeled
- **Explore data structure** — find natural groupings before modeling
- **Feature reduction** — simplify high-dimensional data
- **Customer segmentation** — group customers by behavior
- **Preprocessing** — reduce features before supervised learning

### 1.1 Generating Synthetic Data for Clustering

Let's create several datasets with different structures to test our clustering algorithms.

In [ ]:
# Generate different types of cluster structures
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 1. Well-separated blobs
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='Set1', s=15, alpha=0.7)
axes[0].set_title('Well-separated Blobs')

# 2. Moons (non-convex)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='Set1', s=15, alpha=0.7)
axes[1].set_title('Moons (Non-convex)')

# 3. Anisotropic blobs (stretched)
X_aniso, y_aniso = make_blobs(n_samples=300, centers=3, random_state=42)
transformation = [[0.6, -0.6], [-0.4, 0.8]]  # stretch the clusters
X_aniso = np.dot(X_aniso, transformation)
axes[2].scatter(X_aniso[:, 0], X_aniso[:, 1], c=y_aniso, cmap='Set1', s=15, alpha=0.7)
axes[2].set_title('Anisotropic (Stretched)')

# 4. Blobs with noise
X_noisy, y_noisy = make_blobs(n_samples=250, centers=3, cluster_std=1.5, random_state=42)
# Add uniform noise points
noise = np.random.uniform(-12, 12, (50, 2))
X_noisy = np.vstack([X_noisy, noise])
y_noisy = np.concatenate([y_noisy, np.full(50, -1)])
axes[3].scatter(X_noisy[:, 0], X_noisy[:, 1], c=y_noisy, cmap='Set1', s=15, alpha=0.7)
axes[3].set_title('Blobs with Noise')

for ax in axes:
    ax.set_xlabel('X₁')
    ax.set_ylabel('X₂')

plt.suptitle('Different Cluster Structures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Different algorithms work best on different structures:')
print('  Blobs → k-Means works well')
print('  Moons → DBSCAN works well')
print('  Noise → DBSCAN can identify outliers')

---
## 2. k-Means Clustering

**k-Means** is the most popular clustering algorithm. It partitions data into k clusters by minimizing the within-cluster sum of squared distances.

**Algorithm:**
1. Initialize k cluster centroids randomly
2. **Assign** each point to the nearest centroid
3. **Update** centroids to the mean of assigned points
4. Repeat steps 2-3 until convergence

**Requirements:** Must specify k in advance. Assumes spherical, equally-sized clusters.

### 2.1 k-Means Step by Step

Let's visualize each iteration of the k-Means algorithm to understand how it works.

In [ ]:
# Visualize k-Means iterations on the blobs data
X = X_blobs.copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
max_iters = [1, 2, 3, 5, 10, 50]

for idx, n_iter in enumerate(max_iters):
    ax = axes[idx // 3][idx % 3]
    
    # Run k-Means with limited iterations
    km = KMeans(n_clusters=4, init='random', n_init=1, max_iter=n_iter, random_state=42)
    labels = km.fit_predict(X)
    centers = km.cluster_centers_
    
    # Plot points colored by cluster
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='Set1', s=15, alpha=0.6)
    # Plot centroids
    ax.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=200, 
               edgecolors='white', linewidths=2, zorder=5)
    ax.set_title(f'Iteration {n_iter} (inertia={km.inertia_:.0f})', fontsize=11)

plt.suptitle('k-Means Convergence Process', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('The algorithm converges quickly — centroids stabilize after a few iterations.')
print('Inertia (within-cluster sum of squares) decreases at each step.')

### 2.2 k-Means in Practice

Let's apply k-Means to the Iris dataset (using only features, ignoring true labels) and compare our clusters with the actual species.

In [ ]:
# Load Iris (pretend we don't know the labels)
iris = load_iris()
X_iris = iris.data
y_true = iris.target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# k-Means with k=3 (since we know there are 3 species)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
y_km = km.fit_predict(X_scaled)

# Compare clusters with true labels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Use PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# True labels
for c in range(3):
    mask = y_true == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                     label=iris.target_names[c], s=30, alpha=0.7)
axes[0].set_title('True Species Labels')
axes[0].legend()

# k-Means clusters
for c in range(3):
    mask = y_km == c
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                     label=f'Cluster {c}', s=30, alpha=0.7)
# Plot centroids
centers_pca = pca.transform(km.cluster_centers_)
axes[1].scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X', 
                 s=200, edgecolors='white', linewidths=2, zorder=5)
axes[1].set_title('k-Means Clusters (k=3)')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('k-Means on Iris: Clusters vs True Labels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# External evaluation (when true labels are available)
ari = adjusted_rand_score(y_true, y_km)
nmi = normalized_mutual_info_score(y_true, y_km)
print(f'Adjusted Rand Index: {ari:.3f} (1.0 = perfect match)')
print(f'Normalized Mutual Info: {nmi:.3f} (1.0 = perfect match)')

---
## 3. Choosing k: Elbow Method & Silhouette Analysis

In practice, we often don't know the true number of clusters. Two common methods help us choose k:

### 3.1 The Elbow Method

Plot inertia (within-cluster SSE) vs k. Look for the "elbow" where adding more clusters gives diminishing returns.

In [ ]:
# Elbow method
k_range = range(2, 11)
inertias = []
sil_scores = []
ch_scores = []   # Calinski-Harabasz index
db_scores = []   # Davies-Bouldin index

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Elbow plot (inertia)
axes[0,0].plot(k_range, inertias, 'b-o', markersize=6)
axes[0,0].set_xlabel('k')
axes[0,0].set_ylabel('Inertia (SSE)')
axes[0,0].set_title('Elbow Method')
axes[0,0].axvline(3, color='red', ls='--', alpha=0.5, label='k=3')
axes[0,0].legend()

# 2. Silhouette score (higher is better)
axes[0,1].plot(k_range, sil_scores, 'g-o', markersize=6)
axes[0,1].set_xlabel('k')
axes[0,1].set_ylabel('Silhouette Score')
axes[0,1].set_title('Silhouette Score (higher = better)')
best_k_sil = list(k_range)[np.argmax(sil_scores)]
axes[0,1].axvline(best_k_sil, color='red', ls='--', alpha=0.5, label=f'Best k={best_k_sil}')
axes[0,1].legend()

# 3. Calinski-Harabasz (higher is better)
axes[1,0].plot(k_range, ch_scores, 'r-o', markersize=6)
axes[1,0].set_xlabel('k')
axes[1,0].set_ylabel('Calinski-Harabasz Index')
axes[1,0].set_title('Calinski-Harabasz (higher = better)')

# 4. Davies-Bouldin (lower is better)
axes[1,1].plot(k_range, db_scores, 'm-o', markersize=6)
axes[1,1].set_xlabel('k')
axes[1,1].set_ylabel('Davies-Bouldin Index')
axes[1,1].set_title('Davies-Bouldin (lower = better)')

plt.suptitle('Choosing k: Multiple Criteria (Iris Dataset)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Multiple metrics suggest k=2 or k=3 for Iris.')
print('k=3 matches the true number of species.')

### 3.2 Silhouette Plots

A silhouette plot shows the silhouette coefficient for each sample, grouped by cluster. It reveals cluster quality and size balance.

In [ ]:
def plot_silhouette(X, k, ax):
    """
    Create a silhouette plot for k-Means with k clusters.
    """
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil_avg = silhouette_score(X, labels)
    sil_values = silhouette_samples(X, labels)
    
    y_lower = 10
    colors = plt.cm.Set1(np.linspace(0, 1, k))
    
    for i in range(k):
        cluster_sil = sil_values[labels == i]
        cluster_sil.sort()
        y_upper = y_lower + len(cluster_sil)
        
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                          alpha=0.7, color=colors[i])
        ax.text(-0.05, y_lower + 0.5 * len(cluster_sil), str(i), fontsize=10)
        y_lower = y_upper + 10
    
    ax.axvline(sil_avg, color='red', ls='--', label=f'Avg={sil_avg:.3f}')
    ax.set_title(f'k={k}')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.legend(fontsize=9)

# Compare k=2,3,4,5
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for idx, k in enumerate([2, 3, 4, 5]):
    plot_silhouette(X_scaled, k, axes[idx])

plt.suptitle('Silhouette Analysis for Different k (Iris)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('A good clustering has:')
print('  - High average silhouette score (red dashed line)')
print('  - All clusters above average (no clusters fully below the line)')
print('  - Similar cluster sizes (similar widths)')

---
## 4. Hierarchical Clustering

**Hierarchical clustering** builds a tree (dendrogram) of nested clusters. There are two approaches:

| Approach | Method | Process |
|----------|--------|---------|
| **Agglomerative** (bottom-up) | Start with each point as a cluster, merge closest pairs | Most common |
| **Divisive** (top-down) | Start with one cluster, split recursively | Less common |

**Linkage methods** define how distance between clusters is measured:

| Linkage | Distance between clusters | Tendency |
|---------|--------------------------|----------|
| `ward` | Minimize total within-cluster variance | Compact, equal-sized |
| `complete` | Maximum distance between points | Compact clusters |
| `average` | Average distance between all pairs | Balanced |
| `single` | Minimum distance between points | Can find elongated clusters |

### 4.1 Dendrogram Visualization

The dendrogram shows the hierarchy of merges — we cut it at a chosen height to get k clusters.

In [ ]:
# Compute linkage matrix
Z = linkage(X_scaled, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full dendrogram
dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=30, 
           leaf_rotation=90, leaf_font_size=8)
axes[0].set_title('Dendrogram (Ward Linkage)')
axes[0].set_xlabel('Sample index or (cluster size)')
axes[0].set_ylabel('Distance')
axes[0].axhline(y=7, color='red', ls='--', label='Cut at height=7 → 3 clusters')
axes[0].legend()

# Compare linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
for method in linkage_methods:
    Z_m = linkage(X_scaled, method=method)
    dendrogram(Z_m, ax=axes[1], truncate_mode='level', p=3, 
               no_labels=True, above_threshold_color='gray')
# Just show ward for clarity
axes[1].clear()
for i, method in enumerate(linkage_methods):
    Z_m = linkage(X_scaled[:50], method=method)  # use subset for clarity
    dendrogram(Z_m, ax=axes[1], truncate_mode='level', p=4,
               no_labels=True, color_threshold=0)
    axes[1].set_title('Dendrogram Comparison (first 50 samples)')

# Simpler: just compare ward
axes[1].clear()
dendrogram(linkage(X_scaled[:60], method='ward'), ax=axes[1],
           leaf_rotation=90, leaf_font_size=8)
axes[1].set_title('Ward Linkage (first 60 samples)')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Distance')

plt.tight_layout()
plt.show()

### 4.2 Agglomerative Clustering with scikit-learn

Let's apply agglomerative clustering and compare results with k-Means.

In [ ]:
# Agglomerative clustering
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
y_agg = agg.fit_predict(X_scaled)

# Compare k-Means vs Hierarchical
km = KMeans(n_clusters=3, random_state=42, n_init=10)
y_km = km.fit_predict(X_scaled)

X_pca = PCA(n_components=2).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# True labels
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='Set1', s=25, alpha=0.7)
axes[0].set_title('True Labels')

# k-Means
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_km, cmap='Set1', s=25, alpha=0.7)
axes[1].set_title(f'k-Means (ARI={adjusted_rand_score(y_true, y_km):.3f})')

# Hierarchical
axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=y_agg, cmap='Set1', s=25, alpha=0.7)
axes[2].set_title(f'Hierarchical Ward (ARI={adjusted_rand_score(y_true, y_agg):.3f})')

for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Iris: k-Means vs Hierarchical Clustering', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantitative comparison
print(f'{"Method":<25} {"Silhouette":>12} {"ARI":>8} {"NMI":>8}')
print('-' * 55)
for name, labels in [('k-Means', y_km), ('Hierarchical (Ward)', y_agg)]:
    sil = silhouette_score(X_scaled, labels)
    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)
    print(f'{name:<25} {sil:>12.3f} {ari:>8.3f} {nmi:>8.3f}')

---
## 5. DBSCAN: Density-Based Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups together points that are closely packed and marks low-density regions as noise.

**Key advantages over k-Means:**
- Does NOT require specifying k
- Can find clusters of arbitrary shape
- Naturally identifies outliers/noise

**Parameters:**
- `eps` — radius of neighborhood (critical parameter)
- `min_samples` — minimum points to form a dense region

### 5.1 DBSCAN on Non-Convex Data

DBSCAN excels at finding non-convex clusters where k-Means fails.

In [ ]:
# Compare k-Means vs DBSCAN on the moons dataset
X_moons_s = StandardScaler().fit_transform(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# True labels
axes[0].scatter(X_moons_s[:, 0], X_moons_s[:, 1], c=y_moons, cmap='Set1', s=20)
axes[0].set_title('True Labels')

# k-Means (fails on moons)
y_km = KMeans(n_clusters=2, random_state=42).fit_predict(X_moons_s)
axes[1].scatter(X_moons_s[:, 0], X_moons_s[:, 1], c=y_km, cmap='Set1', s=20)
axes[1].set_title(f'k-Means (ARI={adjusted_rand_score(y_moons, y_km):.3f})')

# DBSCAN (succeeds!)
db = DBSCAN(eps=0.3, min_samples=5)
y_db = db.fit_predict(X_moons_s)
# Color noise points differently
colors = y_db.copy().astype(float)
colors[y_db == -1] = -1  # noise
axes[2].scatter(X_moons_s[y_db >= 0, 0], X_moons_s[y_db >= 0, 1], 
                 c=y_db[y_db >= 0], cmap='Set1', s=20)
if (y_db == -1).sum() > 0:
    axes[2].scatter(X_moons_s[y_db == -1, 0], X_moons_s[y_db == -1, 1],
                     c='black', marker='x', s=30, label='Noise')
    axes[2].legend()
n_clusters = len(set(y_db)) - (1 if -1 in y_db else 0)
axes[2].set_title(f'DBSCAN ({n_clusters} clusters, ARI={adjusted_rand_score(y_moons, y_db):.3f})')

for ax in axes:
    ax.set_xlabel('X₁')
    ax.set_ylabel('X₂')

plt.suptitle('Moons: k-Means vs DBSCAN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('k-Means fails because it assumes convex clusters.')
print('DBSCAN correctly finds the two moon-shaped clusters!')

### 5.2 DBSCAN Parameter Sensitivity

DBSCAN is sensitive to the `eps` parameter. Let's see how different values affect the clustering.

In [ ]:
eps_values = [0.1, 0.2, 0.3, 0.5, 0.8, 1.5]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for idx, eps in enumerate(eps_values):
    ax = axes[idx // 3][idx % 3]
    db = DBSCAN(eps=eps, min_samples=5)
    y_db = db.fit_predict(X_moons_s)
    
    n_clusters = len(set(y_db)) - (1 if -1 in y_db else 0)
    n_noise = (y_db == -1).sum()
    
    # Plot clusters
    ax.scatter(X_moons_s[y_db >= 0, 0], X_moons_s[y_db >= 0, 1],
               c=y_db[y_db >= 0], cmap='Set1', s=15)
    if n_noise > 0:
        ax.scatter(X_moons_s[y_db == -1, 0], X_moons_s[y_db == -1, 1],
                    c='black', marker='x', s=20)
    ax.set_title(f'eps={eps}\n{n_clusters} clusters, {n_noise} noise', fontsize=10)

plt.suptitle('DBSCAN: Effect of eps Parameter', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Too small eps → too many clusters, many noise points')
print('Too large eps → everything merged into one cluster')
print('Tip: Use k-distance plot (nearest neighbor distances) to choose eps.')

---
## 6. PCA: Principal Component Analysis

**PCA** finds the directions (principal components) of maximum variance in the data. It's the most important dimensionality reduction technique.

**What PCA does:**
1. Center the data (subtract mean)
2. Find directions of maximum variance (eigenvectors of covariance matrix)
3. Project data onto top-k directions

**Key properties:**
- Components are orthogonal (uncorrelated)
- Ordered by explained variance (PC1 > PC2 > ...)
- Linear transformation — fast and deterministic

### 6.1 PCA on the Iris Dataset

Let's reduce the 4-dimensional Iris data to 2 dimensions and see how much information is preserved.

In [ ]:
# PCA on scaled Iris data
pca = PCA()  # fit all components
X_pca_all = pca.fit_transform(X_scaled)

# Explained variance
print('PCA Results:')
print(f'{"Component":<12} {"Var Explained":>15} {"Cumulative":>12}')
print('-' * 42)
cumvar = 0
for i, (var, ratio) in enumerate(zip(pca.explained_variance_, 
                                      pca.explained_variance_ratio_)):
    cumvar += ratio
    print(f'PC{i+1:<10} {ratio:>14.1%} {cumvar:>11.1%}')

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 5), pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
axes[0].plot(range(1, 5), np.cumsum(pca.explained_variance_ratio_), 'ro-')
axes[0].axhline(0.95, color='gray', ls='--', alpha=0.5, label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].legend()

# 2D projection
for c in range(3):
    mask = y_true == c
    axes[1].scatter(X_pca_all[mask, 0], X_pca_all[mask, 1],
                     label=iris.target_names[c], s=30, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[1].set_title('Iris: 2D PCA Projection')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nPC1 + PC2 capture {sum(pca.explained_variance_ratio_[:2]):.1%} of the total variance.')
print('We can reduce 4D to 2D with minimal information loss!')

### 6.2 Understanding PCA Components

Each principal component is a linear combination of original features. The loadings tell us which features contribute most.

In [ ]:
# PCA loadings (component weights)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(4)],
    index=iris.feature_names
)
print('PCA Loadings (feature weights for each component):')
print(loadings.round(3))

# Visualize loadings as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(loadings, annot=True, cmap='RdBu_r', center=0, ax=axes[0], fmt='.2f')
axes[0].set_title('PCA Loadings Heatmap')

# Biplot: show both data points and feature vectors
for c in range(3):
    mask = y_true == c
    axes[1].scatter(X_pca_all[mask, 0], X_pca_all[mask, 1],
                     label=iris.target_names[c], s=15, alpha=0.5)

# Add feature vectors
scale = 3  # scale arrows for visibility
for i, feature in enumerate(iris.feature_names):
    axes[1].arrow(0, 0, pca.components_[0, i]*scale, pca.components_[1, i]*scale,
                   head_width=0.1, head_length=0.05, fc='red', ec='red')
    axes[1].text(pca.components_[0, i]*scale*1.15, pca.components_[1, i]*scale*1.15,
                  feature, color='red', fontsize=9, ha='center')

axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PCA Biplot')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Biplot interpretation:')
print('  - Arrow direction: which PC the feature contributes to')
print('  - Arrow length: how much the feature contributes')
print('  - Arrows pointing same direction: correlated features')

---
## 7. PCA for Visualization & Dimensionality Reduction

PCA has two main practical uses: making high-dimensional data visualizable and improving model performance by reducing features.

### 7.1 PCA vs t-SNE for Visualization

While PCA preserves global structure, **t-SNE** better preserves local neighborhoods — making it excellent for visualization.

In [ ]:
# Compare PCA and t-SNE on the Wine dataset
wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)

# PCA
X_pca = PCA(n_components=2).fit_transform(X_wine)

# t-SNE (stochastic — takes longer)
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_wine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for c in range(3):
    mask = wine.target == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                     label=wine.target_names[c], s=30, alpha=0.7)
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                     label=wine.target_names[c], s=30, alpha=0.7)

axes[0].set_title('PCA (preserves global structure)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

axes[1].set_title('t-SNE (preserves local structure)')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend()

plt.suptitle('Wine Dataset: PCA vs t-SNE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('PCA: fast, deterministic, preserves distances, invertible')
print('t-SNE: slow, stochastic, better cluster separation, not invertible')
print('Use PCA for analysis, t-SNE for visualization only.')

### 7.2 PCA as Preprocessing for Supervised Learning

Reducing dimensions before classification can improve performance by removing noise and collinearity.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Compare classification with and without PCA
X_train, X_test, y_train, y_test = train_test_split(
    X_wine, wine.target, test_size=0.2, random_state=42, stratify=wine.target
)

n_components_range = range(1, 14)
results = []

for n_comp in n_components_range:
    # PCA + Random Forest pipeline
    pipe = Pipeline([
        ('pca', PCA(n_components=n_comp)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5)
    pipe.fit(X_train, y_train)
    
    results.append({
        'n_components': n_comp,
        'cv_mean': scores.mean(),
        'cv_std': scores.std(),
        'test': pipe.score(X_test, y_test)
    })

df_pca_results = pd.DataFrame(results)

# Also get score without PCA
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
full_cv = cross_val_score(rf_full, X_train, y_train, cv=5).mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_pca_results['n_components'], df_pca_results['cv_mean'], 'b-o', label='PCA + RF (CV)')
ax.fill_between(df_pca_results['n_components'],
                 df_pca_results['cv_mean'] - df_pca_results['cv_std'],
                 df_pca_results['cv_mean'] + df_pca_results['cv_std'], alpha=0.15)
ax.axhline(full_cv, color='red', ls='--', label=f'RF without PCA: {full_cv:.3f}')
ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('CV Accuracy')
ax.set_title('Effect of PCA on Classification Performance')
ax.legend()
plt.tight_layout()
plt.show()

best_row = df_pca_results.loc[df_pca_results['cv_mean'].idxmax()]
print(f'Best with PCA: {best_row["n_components"]:.0f} components, CV={best_row["cv_mean"]:.3f}')
print(f'Without PCA (13 features): CV={full_cv:.3f}')
print(f'PCA can maintain or improve accuracy with fewer features!')

---
## 8. Advanced Model Evaluation: Bias-Variance Trade-off

The **bias-variance trade-off** is the fundamental tension in machine learning:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

| Concept | Meaning | Caused by |
|---------|---------|----------|
| **High Bias** | Model too simple, misses patterns | Underfitting |
| **High Variance** | Model too complex, fits noise | Overfitting |

The sweet spot is a model complex enough to capture patterns but simple enough to generalize.

### 8.1 Visualizing Bias-Variance Trade-off

Let's see how model complexity affects bias and variance using polynomial regression as an example.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Generate noisy data from a true function
np.random.seed(42)
n = 30
X_reg = np.sort(np.random.uniform(0, 1, n)).reshape(-1, 1)
y_true_func = np.sin(2 * np.pi * X_reg).ravel()  # true function
y_reg = y_true_func + np.random.normal(0, 0.3, n)  # noisy observations

X_plot = np.linspace(0, 1, 200).reshape(-1, 1)
y_true_plot = np.sin(2 * np.pi * X_plot).ravel()

degrees = [1, 3, 5, 15]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, d in enumerate(degrees):
    poly = PolynomialFeatures(degree=d)
    X_poly = poly.fit_transform(X_reg)
    X_plot_poly = poly.transform(X_plot)
    
    lr = LinearRegression()
    lr.fit(X_poly, y_reg)
    y_pred = lr.predict(X_plot_poly)
    
    train_mse = mean_squared_error(y_reg, lr.predict(X_poly))
    
    axes[idx].scatter(X_reg, y_reg, c='blue', s=25, alpha=0.7, label='Data')
    axes[idx].plot(X_plot, y_true_plot, 'g--', lw=1.5, label='True function')
    axes[idx].plot(X_plot, y_pred, 'r-', lw=2, label='Fitted')
    axes[idx].set_title(f'Degree {d}\nTrain MSE={train_mse:.3f}', fontsize=11)
    axes[idx].set_ylim(-2, 2)
    if idx == 0:
        axes[idx].legend(fontsize=8)

plt.suptitle('Bias-Variance Trade-off: Polynomial Regression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Degree 1: High bias (underfitting) — too simple')
print('Degree 3-5: Good balance — captures the pattern')
print('Degree 15: High variance (overfitting) — fits noise')

---
## 9. Learning Curves & Validation Curves

Two diagnostic plots help diagnose overfitting/underfitting:

- **Learning curve**: performance vs training set size
- **Validation curve**: performance vs hyperparameter value

### 9.1 Learning Curves

A learning curve shows how model performance changes as we increase the training set size.

In [ ]:
from sklearn.model_selection import learning_curve

wine = load_wine()
X_wine_s = StandardScaler().fit_transform(wine.data)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
models = {
    'kNN (k=1) — High Variance': KNeighborsClassifier(n_neighbors=1),
    'Random Forest — Good Balance': RandomForestClassifier(n_estimators=100, random_state=42),
    'kNN (k=50) — High Bias': KNeighborsClassifier(n_neighbors=50),
}

for idx, (name, model) in enumerate(models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_wine_s, wine.target, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    axes[idx].plot(train_sizes, train_mean, 'b-o', markersize=4, label='Training')
    axes[idx].fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='blue')
    axes[idx].plot(train_sizes, val_mean, 'r-s', markersize=4, label='Validation')
    axes[idx].fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1, color='red')
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('Training Set Size')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].legend(fontsize=8)
    axes[idx].set_ylim(0.5, 1.05)

plt.suptitle('Learning Curves: Diagnosing Bias and Variance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('How to read learning curves:')
print('  High Variance: large gap between train (high) and val (low)')
print('  High Bias: both curves plateau at low score')
print('  Good Model: both curves converge at high score')

### 9.2 Validation Curves

A validation curve shows how a hyperparameter affects training and validation performance — helping us find the sweet spot.

In [ ]:
from sklearn.model_selection import validation_curve

# Validation curve for kNN (varying k)
k_range = range(1, 40)
train_scores, val_scores = validation_curve(
    KNeighborsClassifier(), X_wine_s, wine.target,
    param_name='n_neighbors', param_range=k_range,
    cv=5, scoring='accuracy'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# kNN validation curve
axes[0].plot(k_range, train_scores.mean(axis=1), 'b-o', markersize=3, label='Training')
axes[0].plot(k_range, val_scores.mean(axis=1), 'r-s', markersize=3, label='Validation')
axes[0].fill_between(k_range, train_scores.mean(axis=1)-train_scores.std(axis=1),
                      train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.1, color='blue')
axes[0].fill_between(k_range, val_scores.mean(axis=1)-val_scores.std(axis=1),
                      val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.1, color='red')
best_k = list(k_range)[np.argmax(val_scores.mean(axis=1))]
axes[0].axvline(best_k, color='green', ls='--', alpha=0.5, label=f'Best k={best_k}')
axes[0].set_xlabel('k (number of neighbors)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('kNN: Validation Curve')
axes[0].legend()
axes[0].annotate('Overfitting\n(low k)', xy=(2, 0.92), fontsize=9, color='gray')
axes[0].annotate('Underfitting\n(high k)', xy=(32, 0.75), fontsize=9, color='gray')

# Random Forest: n_estimators validation curve
n_range = [5, 10, 25, 50, 100, 200, 500]
train_scores2, val_scores2 = validation_curve(
    RandomForestClassifier(random_state=42), X_wine_s, wine.target,
    param_name='n_estimators', param_range=n_range,
    cv=5, scoring='accuracy'
)

axes[1].plot(n_range, train_scores2.mean(axis=1), 'b-o', markersize=4, label='Training')
axes[1].plot(n_range, val_scores2.mean(axis=1), 'r-s', markersize=4, label='Validation')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Random Forest: Validation Curve')
axes[1].legend()
axes[1].set_xscale('log')

plt.suptitle('Validation Curves: Tuning Hyperparameters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Practical Example: Customer Segmentation

A retail company wants to segment their customers based on purchasing behavior. We'll use the full unsupervised learning pipeline: preprocess → reduce dimensions → cluster → analyze.

### Step 1: Generate Customer Data

In [ ]:
# Generate synthetic customer data
np.random.seed(42)
n = 500

# 4 customer segments (hidden)
segments = np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.25, 0.25, 0.2])

data = []
for seg in segments:
    if seg == 0:   # Budget shoppers: low spend, high frequency, low recency
        r = {'recency': np.random.normal(10, 5), 'frequency': np.random.normal(20, 5),
             'monetary': np.random.normal(30, 10), 'avg_basket': np.random.normal(15, 5),
             'num_categories': np.random.poisson(3), 'discount_usage': np.random.normal(0.7, 0.15)}
    elif seg == 1: # Premium: high spend, moderate frequency
        r = {'recency': np.random.normal(15, 8), 'frequency': np.random.normal(12, 4),
             'monetary': np.random.normal(200, 50), 'avg_basket': np.random.normal(80, 20),
             'num_categories': np.random.poisson(6), 'discount_usage': np.random.normal(0.2, 0.1)}
    elif seg == 2: # Occasional: high recency, low frequency
        r = {'recency': np.random.normal(60, 20), 'frequency': np.random.normal(3, 2),
             'monetary': np.random.normal(50, 20), 'avg_basket': np.random.normal(40, 15),
             'num_categories': np.random.poisson(2), 'discount_usage': np.random.normal(0.4, 0.2)}
    else:          # Loyal: low recency, high frequency, high spend
        r = {'recency': np.random.normal(5, 3), 'frequency': np.random.normal(25, 6),
             'monetary': np.random.normal(150, 40), 'avg_basket': np.random.normal(60, 15),
             'num_categories': np.random.poisson(8), 'discount_usage': np.random.normal(0.3, 0.1)}
    data.append(r)

df_cust = pd.DataFrame(data)
# Clip to realistic ranges
df_cust['recency'] = df_cust['recency'].clip(1, 120)
df_cust['frequency'] = df_cust['frequency'].clip(1, 50).astype(int)
df_cust['monetary'] = df_cust['monetary'].clip(5, 500)
df_cust['avg_basket'] = df_cust['avg_basket'].clip(5, 200)
df_cust['num_categories'] = df_cust['num_categories'].clip(1, 15)
df_cust['discount_usage'] = df_cust['discount_usage'].clip(0, 1)

print(f'Customer dataset: {df_cust.shape}')
print(df_cust.describe().round(2))

### Step 2: Preprocessing & Optimal k

Scale the features and use the elbow method to choose the number of segments.

In [ ]:
# Scale features
scaler = StandardScaler()
X_cust = scaler.fit_transform(df_cust)

# Find optimal k
k_range = range(2, 10)
metrics = {'inertia': [], 'silhouette': [], 'calinski': []}

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cust)
    metrics['inertia'].append(km.inertia_)
    metrics['silhouette'].append(silhouette_score(X_cust, labels))
    metrics['calinski'].append(calinski_harabasz_score(X_cust, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(k_range, metrics['inertia'], 'b-o')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(k_range, metrics['silhouette'], 'g-o')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')

axes[2].plot(k_range, metrics['calinski'], 'r-o')
axes[2].set_title('Calinski-Harabasz')
axes[2].set_xlabel('k')

plt.suptitle('Choosing Number of Customer Segments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Metrics suggest k=4 (matches our hidden segments).')

### Step 3: Cluster and Analyze

Apply k-Means with k=4 and profile each segment.

In [ ]:
# Final clustering
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cust['segment'] = km.fit_predict(X_cust)

# Segment profiles
profiles = df_cust.groupby('segment').agg(
    count=('recency', 'size'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_basket=('avg_basket', 'mean'),
    avg_categories=('num_categories', 'mean'),
    avg_discount=('discount_usage', 'mean')
).round(1)

print('=== Customer Segment Profiles ===')
print(profiles.to_string())

# Name the segments based on profiles
segment_names = {}
for seg in range(4):
    p = profiles.loc[seg]
    if p['avg_monetary'] > 150 and p['avg_frequency'] > 20:
        segment_names[seg] = 'Loyal Premium'
    elif p['avg_monetary'] > 150:
        segment_names[seg] = 'High Spenders'
    elif p['avg_recency'] > 40:
        segment_names[seg] = 'At Risk'
    elif p['avg_discount'] > 0.5:
        segment_names[seg] = 'Bargain Hunters'
    else:
        segment_names[seg] = f'Segment {seg}'

df_cust['segment_name'] = df_cust['segment'].map(segment_names)
print(f'\nSegment names: {segment_names}')

### Step 4: Visualization Dashboard

Create a comprehensive visualization of the customer segments.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. PCA 2D plot of segments
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cust)
for seg in range(4):
    mask = df_cust['segment'] == seg
    name = segment_names[seg]
    axes[0,0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, s=20, alpha=0.6)
axes[0,0].set_title('Customer Segments (PCA 2D)')
axes[0,0].set_xlabel('PC1')
axes[0,0].set_ylabel('PC2')
axes[0,0].legend(fontsize=9)

# 2. Segment sizes
seg_counts = df_cust['segment_name'].value_counts()
colors = plt.cm.Set2(range(4))
axes[0,1].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
               colors=colors, startangle=90)
axes[0,1].set_title('Segment Distribution')

# 3. Radar chart of segment profiles (normalized)
features_radar = ['avg_recency', 'avg_frequency', 'avg_monetary', 
                   'avg_basket', 'avg_categories', 'avg_discount']
# Normalize profiles to [0,1] for radar
profiles_norm = profiles[features_radar].copy()
for col in features_radar:
    profiles_norm[col] = (profiles_norm[col] - profiles_norm[col].min()) / (
        profiles_norm[col].max() - profiles_norm[col].min())

# Box plot of monetary by segment
df_cust.boxplot(column='monetary', by='segment_name', ax=axes[1,0])
axes[1,0].set_title('Monetary Value by Segment')
axes[1,0].set_xlabel('Segment')
axes[1,0].set_ylabel('Monetary ($)')
plt.sca(axes[1,0])
plt.xticks(rotation=15)
plt.title('Monetary Value by Segment')

# 4. Frequency vs Monetary scatter
for seg in range(4):
    mask = df_cust['segment'] == seg
    name = segment_names[seg]
    axes[1,1].scatter(df_cust.loc[mask, 'frequency'], 
                       df_cust.loc[mask, 'monetary'],
                       label=name, s=20, alpha=0.6)
axes[1,1].set_xlabel('Frequency')
axes[1,1].set_ylabel('Monetary ($)')
axes[1,1].set_title('Frequency vs Monetary by Segment')
axes[1,1].legend(fontsize=9)

plt.suptitle('Customer Segmentation Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 5: Business Recommendations

Let's translate segment insights into actionable marketing strategies.

In [ ]:
print('=' * 65)
print('   CUSTOMER SEGMENTATION — STRATEGIC RECOMMENDATIONS')
print('=' * 65)

for seg in range(4):
    name = segment_names[seg]
    p = profiles.loc[seg]
    count = int(p['count'])
    pct = count / len(df_cust) * 100
    
    print(f'\n--- {name} ({count} customers, {pct:.0f}%) ---')
    print(f'  Profile: Recency={p["avg_recency"]:.0f}d, '
          f'Freq={p["avg_frequency"]:.0f}, '
          f'Spend=${p["avg_monetary"]:.0f}')
    
    if 'Loyal' in name or 'Premium' in name:
        print('  Strategy: VIP treatment, early access, loyalty rewards')
        print('  Goal: Retain and increase share of wallet')
    elif 'High' in name:
        print('  Strategy: Personalized recommendations, premium products')
        print('  Goal: Increase frequency and loyalty')
    elif 'Risk' in name:
        print('  Strategy: Win-back campaigns, special offers, surveys')
        print('  Goal: Re-engage before they leave')
    elif 'Bargain' in name:
        print('  Strategy: Targeted promotions, bundle deals')
        print('  Goal: Increase basket size and reduce discount dependency')
    else:
        print('  Strategy: Further analysis needed')

# Recovery comparison with true segments
ari = adjusted_rand_score(segments, df_cust['segment'])
print(f'\n{"="*65}')
print(f'Recovery accuracy (ARI vs true segments): {ari:.3f}')
print('=' * 65)

---
## 11. Summary

| Section | Key Concepts | Python Tools |
|---------|-------------|---------------|
| 1. Unsupervised Intro | No labels, clustering vs reduction | Conceptual |
| 2. k-Means | Iterative centroid assignment, inertia | `KMeans` |
| 3. Choosing k | Elbow, silhouette, CH, DB indices | `silhouette_score`, `calinski_harabasz_score` |
| 4. Hierarchical | Dendrogram, linkage methods, agglomerative | `AgglomerativeClustering`, `dendrogram` |
| 5. DBSCAN | Density-based, noise detection, eps | `DBSCAN` |
| 6. PCA | Variance maximization, scree plot, loadings | `PCA` |
| 7. PCA Applications | Visualization, preprocessing, t-SNE comparison | `PCA`, `TSNE` |
| 8. Bias-Variance | Trade-off, under/overfitting | Polynomial regression demo |
| 9. Diagnostic Curves | Learning curves, validation curves | `learning_curve`, `validation_curve` |
| 10. Customer Segmentation | End-to-end pipeline, profiling, recommendations | Full workflow |

## Homework

### Task 1: Clustering Comparison
Using `sklearn.datasets.make_blobs(n_samples=500, centers=5, cluster_std=[1.0, 1.5, 0.5, 2.0, 1.0])`:
1. Apply k-Means, Hierarchical (ward), and DBSCAN.
2. Compare using silhouette score and ARI (true labels are available).
3. Visualize all three results side by side.
4. Which method works best and why?

### Task 2: PCA Deep Dive
Using the Wine dataset (13 features):
1. Apply PCA and create a scree plot.
2. How many components are needed to explain 95% of variance?
3. Create a biplot showing feature loadings on PC1-PC2.
4. Compare Random Forest accuracy with 2, 5, 8, and 13 (all) PCA components.

### Task 3: Learning Curve Analysis
Using `sklearn.datasets.load_breast_cancer()`:
1. Plot learning curves for: kNN (k=1), kNN (k=20), SVM (RBF), and Random Forest.
2. Identify which models show overfitting and which show underfitting.
3. For the overfitting model, what would you do to fix it?

### Task 4: End-to-End Segmentation
Create your own synthetic dataset with at least 5 features and 3 hidden segments:
1. Generate the data without revealing segments.
2. Apply the full pipeline: scale → choose k → cluster → PCA visualization.
3. Profile each segment with summary statistics.
4. Create a dashboard with at least 4 plots.
5. Write business recommendations for each segment.

---
## Next Week Preview

**Week 16: Deep Learning & LLMs** — We'll explore neural networks, deep learning fundamentals with PyTorch/Keras, and the basics of Large Language Models (LLMs) that power modern AI assistants.

---
*Applied Statistics with Python (2026) | Week 15 | Unsupervised Learning & Model Evaluation*